# Test Evaluation

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

from utils.utils import load_dataset, evaluate_with_profit

MODEL_DIR = Path("models")

In [ ]:
# quarto preview 04_evaluation.ipynb --to pdf
# quarto render 04_evaluation.ipynb
# black 04_evaluation.ipynb

In [2]:
# Global artifacts
global_model = joblib.load(MODEL_DIR / "global_model.joblib")
global_calibrator = joblib.load(MODEL_DIR / "global_calibrator.joblib")
global_metadata = joblib.load(MODEL_DIR / "global_metadata.joblib")
global_threshold = joblib.load(MODEL_DIR / "global_threshold.joblib")

In [3]:
# Cluster artifacts
cluster_models = {}
cluster_calibrators = {}
cluster_metadata = {}

cluster_thresholds = joblib.load(MODEL_DIR / "cluster_thresholds.joblib")

for cluster_id in cluster_thresholds.keys():
    cluster_models[cluster_id] = joblib.load(MODEL_DIR / f"c{cluster_id}_model.joblib")
    cluster_calibrators[cluster_id] = joblib.load(
        MODEL_DIR / f"c{cluster_id}_calibrator.joblib"
    )
    cluster_metadata[cluster_id] = joblib.load(
        MODEL_DIR / f"c{cluster_id}_metadata.joblib"
    )

In [4]:
X_test_full, y_test = load_dataset("02", "test")

cluster_col = "cluster"

# Keep cluster labels for segmentation
X_test = X_test_full.drop(columns=[cluster_col])


test
X shape: (9043, 60)
y shape: (9043,)

X head:


,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,cat__job_management,cat__job_retired,cat__job_self-employed,cat__job_services,cat__job_student,cat__job_technician,...,bin__housing,bin__loan,bin__poutcome_success,bin__contact_is_unknown,bin__poutcome_is_unknown,bin__multiple_contacts_current_campaign,bin__previous_and_many_current_contacts,bin__many_contacts_and_housing_loan,bin__both_housing_and_personal_loan,cluster
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0


In [5]:
C_call = 1.0
B_sub = 10.0

## 1. Overall Comparison

### 1. Global Model Evaluation

In [6]:
# Align features
X_test_global = X_test[global_metadata["features"]]

# Predict probabilities (raw + calibrated)
global_probs_raw = global_model.predict_proba(X_test_global)[:, 1]
global_probs_cal = global_calibrator.transform(global_probs_raw)

In [7]:
global_metrics = evaluate_with_profit(
    y_true=y_test,
    y_prob=global_probs_cal,
    threshold=global_threshold,
    C_call=C_call,
    B_sub=B_sub,
)

global_results_df = pd.DataFrame([global_metrics], index=["Global"])
display(global_results_df)

,AUC_PR,Precision,Recall,F1,% Contacted,Expected Profit
Global,0.419732,0.264336,0.714556,0.385911,31.626673,5158.96754


The question that is answered by this is: if I use ONE model for everyone, how do I perform?

### 2. Segmented Model Evaluation

We apply a different model per cluster, then recombine results

In [8]:
all_probs_cluster = np.zeros(len(y_test))
y_pred_segmented = np.zeros(len(y_test))

In [9]:
for cluster_id in cluster_thresholds.keys():

    mask = X_test_full[cluster_col] == cluster_id

    X_k = X_test_full[mask].drop(columns=[cluster_col])
    y_k = y_test[mask]

    # align features
    X_k = X_k[cluster_metadata[cluster_id]["features"]]

    model_k = cluster_models[cluster_id]
    calibrator_k = cluster_calibrators[cluster_id]
    threshold_k = cluster_thresholds[cluster_id]

    probs_k = calibrator_k.transform(model_k.predict_proba(X_k)[:, 1])

    all_probs_cluster[mask] = probs_k
    y_pred_segmented[mask] = (probs_k >= threshold_k).astype(int)

In [10]:
# evaluate segmented model
segmented_metrics = {
    "AUC_PR": average_precision_score(y_test, all_probs_cluster),
    "Precision": precision_score(y_test, y_pred_segmented, zero_division=0),
    "Recall": recall_score(y_test, y_pred_segmented),
    "F1": f1_score(y_test, y_pred_segmented),
    "% Contacted": y_pred_segmented.mean() * 100,
    "Expected Profit": np.sum(
        all_probs_cluster[y_pred_segmented == 1] * B_sub - C_call
    ),
}

segmented_results_df = pd.DataFrame([segmented_metrics], index=["Segmented"])
display(segmented_results_df)

,AUC_PR,Precision,Recall,F1,% Contacted,Expected Profit
Segmented,0.398141,0.272295,0.649338,0.383692,27.900033,4833.213341


The question that is answered by this is: if I specialize by cluster, how do I perform overall?

### 3. Comparison

In [11]:
final_comparison = pd.concat([global_results_df, segmented_results_df])

final_comparison["Expected Profit"] = final_comparison["Expected Profit"].round(0)
final_comparison["% Contacted"] = final_comparison["% Contacted"].round(2)

display(final_comparison)

,AUC_PR,Precision,Recall,F1,% Contacted,Expected Profit
Global,0.419732,0.264336,0.714556,0.385911,31.63,5159.0
Segmented,0.398141,0.272295,0.649338,0.383692,27.90,4833.0


The question that is answered by this is: which strategy wins overall?

## 2. Per-cluster comparison

In [ ]:
comparison_by_cluster = []

for cluster_id in sorted(cluster_thresholds.keys()):
    mask = X_test_full[cluster_col] == cluster_id

    y_k = y_test[mask]

    # ===== GLOBAL MODEL (same model, evaluated inside cluster) =====
    probs_g = global_probs_cal[mask]
    # pred_g = (probs_g >= global_threshold).astype(int)

    metrics_g = evaluate_with_profit(
        y_true=y_k,
        y_prob=probs_g,
        threshold=global_threshold,
        C_call=C_call,
        B_sub=B_sub,
    )

    # ===== SEGMENTED MODEL (cluster-specific) =====
    probs_k = all_probs_cluster[mask]
    # pred_k = (probs_k >= cluster_thresholds[cluster_id]).astype(int)

    metrics_k = evaluate_with_profit(
        y_true=y_k,
        y_prob=probs_k,
        threshold=cluster_thresholds[cluster_id],
        C_call=C_call,
        B_sub=B_sub,
    )

    comparison_by_cluster.append(
        {
            "Cluster": cluster_id,
            "Global Profit": metrics_g["Expected Profit"],
            "Segmented Profit": metrics_k["Expected Profit"],
            "Global % Contacted": metrics_g["% Contacted"],
            "Segmented % Contacted": metrics_k["% Contacted"],
        }
    )

comparison_by_cluster_df = pd.DataFrame(comparison_by_cluster).set_index("Cluster")

comparison_by_cluster_df["Global Profit"] = comparison_by_cluster_df[
    "Global Profit"
].round(0)
comparison_by_cluster_df["Segmented Profit"] = comparison_by_cluster_df[
    "Segmented Profit"
].round(0)

display(comparison_by_cluster_df)

The question that is answered by this is: does segmentation actually improve results WHERE it matters? 